# Chapter 19 --- Threshold Calibration and Cross-Domain Transfer

This notebook accompanies Chapter 19 of *How to Build Policy-Governed AI Agents*.
We demonstrate that the governance architecture is **domain-portable** --- only
the threshold vector $\Theta = (\theta_{\mathrm{sim}}, \theta_{\mathrm{margin}},
\theta_{\mathrm{esc}}, D_{\max}, B)$ needs recalibration when transferring
to a new domain.

**Prerequisites:** Chapters 2 (GMS fundamentals), 8 (claim verification), 15 (release gates).

**Library module built:** `governed_agent/calibration.py`

In [ ]:
# === Preamble (all notebooks) ===

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import random
random.seed(42)
np.random.seed(42)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 10})

## Key Concepts

| Term | Definition |
|------|------------|
| **Threshold vector** | $\Theta = (\theta_{\mathrm{sim}},\; \theta_{\mathrm{margin}},\; \theta_{\mathrm{esc}},\; D_{\max},\; B)$ --- the five parameters that convert raw geometric signals into binary pass/fail verification decisions. |
| **Design of Experiments (DOE)** | Statistical discipline of planning experiments to maximize information per run. Here we treat threshold tuning as a DOE problem: parameters are factors, configurations are runs, accuracy is the response. |
| **False accept (FA)** | The verifier incorrectly lets an erroneous claim pass. Caused by thresholds that are too loose. |
| **False reject (FR)** | The verifier incorrectly flags a correct claim as failing. Caused by thresholds that are too tight. |
| **Sensitivity analysis** | Measures each parameter's *impact score* (max - min accuracy across its levels). High impact = strong lever on verification performance. |
| **Factorial grid search** | Evaluates every combination of chosen parameter levels (e.g. $5 \times 4 = 20$ configurations). |
| **Claim types** | FACT (relational triple check via `query_triples`/`score_triple`), NUMERIC (value comparison within $\theta_{\mathrm{margin}}$), THRESHOLD (metric vs. bound). |
| **Domain portability** | The governance architecture transfers across domains (classification to regression) by recalibrating thresholds and swapping the knowledge document; all verifier code remains identical. |

---

## Lab 19.1 --- DOE on Threshold Parameters

We treat threshold calibration as a design-of-experiments (DOE) problem.
Each threshold parameter is a factor; each configuration is an
experimental run. The response variable is verification accuracy on a
labeled claim set.

### Step 1: Build the Knowledge Store and Train Models on Iris

In [ ]:
# --- Build the Iris knowledge store ---
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document
from knowlytix.core.config import GeometryConfig, TrainConfig

MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(
    store_path="governance_monograph/lab_stores/ch19_calibration",
    geometry=GeometryConfig(m=128),
    train=TrainConfig(epochs=200, device=str(DEVICE)),
)

store = GMSExpertStore(config, device=DEVICE)
if not store.load():
    print("Building store from scratch (200 epochs on GPU)...")
    result = ingest_document(store, MD_PATH, llm=None,
                             config=config, device=DEVICE)
    store.save()

stats = store.stats()
print("\n=== Iris Knowledge Store ===")
print(f"  Entities:  {stats['entities']}")
print(f"  Relations: {stats['relations']}")
print(f"  Triples:   {stats['triples']}")
print(f"  ENM:       {stats['enm_entries']}")

In [ ]:
# --- Train three classification models on Iris ---
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from knowlytix.knowledge.learn import RuntimeLearner
from knowlytix.knowledge.extract import ExtractedTriple

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42)

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=100, random_state=42),
    "LogisticRegression": LogisticRegression(
        max_iter=200, random_state=42),
}

metrics = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    metrics[name] = {"accuracy": round(acc, 6), "f1": round(f1, 6)}
    print(f"  {name:25s}  accuracy={acc:.4f}  f1={f1:.4f}")

In [4]:
# --- Record metrics in GMS via RuntimeLearner ---
learner = RuntimeLearner(store, config.learn)

training_triples = []
for model_name, model_metrics in metrics.items():
    for metric_name, value in model_metrics.items():
        training_triples.append(ExtractedTriple(
            head=model_name, relation=f"has_{metric_name}",
            tail=f"{value}", numeric_value=value))
    training_triples.append(ExtractedTriple(
        head=model_name, relation="trained_on",
        tail="iris_dataset"))

result = learner.learn_triples(training_triples, device=DEVICE)
store.save()
print(f"RuntimeLearner accepted: {result.accepted}")
print(f"  Triples added: {result.n_triples_added}")
print(f"  Training queued: {result.training_queued}")

  Store saved to governance_monograph/lab_stores/ch19_calibration
RuntimeLearner accepted: True
  Triples added: 9
  Training queued: True


### Step 2: Build Labeled Test Claims

We create a mix of **correct claims** (expected to pass) and
**incorrect claims** (expected to fail). This labeled set serves
as the calibration benchmark.

In [ ]:
from knowlytix.harness.governance.verifier import ClaimType, Claim
from knowlytix.harness.governance.calibration import (
    LabeledClaim,
    ThresholdCalibrator,
)

# --- Correct claims (should pass) ---
correct_claims = [
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="RandomForest was trained on iris dataset",
                    metadata={"head": "RandomForest",
                              "relation": "trained_on",
                              "tail": "iris_dataset"}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="RandomForest test accuracy is 1.0",
                    metadata={"category": "has_accuracy",
                              "entity_id": "RandomForest/1.0",
                              "claimed_value": 1.0}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="GradientBoosting test accuracy is 1.0",
                    metadata={"category": "has_accuracy",
                              "entity_id": "GradientBoosting/1.0",
                              "claimed_value": 1.0}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="LogisticRegression test accuracy is 1.0",
                    metadata={"category": "has_accuracy",
                              "entity_id": "LogisticRegression/1.0",
                              "claimed_value": 1.0}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.THRESHOLD,
                    text="GradientBoosting accuracy meets 0.90 minimum",
                    metadata={"category": "has_accuracy",
                              "entity_id": "GradientBoosting/1.0",
                              "operator": "ge", "limit": 0.90}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.THRESHOLD,
                    text="RandomForest accuracy meets 0.95 minimum",
                    metadata={"category": "has_accuracy",
                              "entity_id": "RandomForest/1.0",
                              "operator": "ge", "limit": 0.95}),
        expected_pass=True),
]

# --- Incorrect claims (should fail) ---
incorrect_claims = [
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="RandomForest accuracy is 0.85 (WRONG)",
                    metadata={"category": "has_accuracy",
                              "entity_id": "RandomForest/1.0",
                              "claimed_value": 0.85}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="GradientBoosting accuracy is 0.92 (WRONG)",
                    metadata={"category": "has_accuracy",
                              "entity_id": "GradientBoosting/1.0",
                              "claimed_value": 0.92}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="RandomForest requires StandardScaler (WRONG)",
                    metadata={"head": "RandomForest",
                              "relation": "requires",
                              "tail": "StandardScaler"}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="LogisticRegression is a tree model (WRONG)",
                    metadata={"head": "LogisticRegression",
                              "relation": "is_type",
                              "tail": "tree_model"}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.THRESHOLD,
                    text="LogisticRegression accuracy exceeds 1.05 (WRONG)",
                    metadata={"category": "has_accuracy",
                              "entity_id": "LogisticRegression/1.0",
                              "operator": "ge", "limit": 1.05}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="SVM was trained on iris (WRONG: not trained)",
                    metadata={"head": "SVM",
                              "relation": "trained_on",
                              "tail": "iris_dataset"}),
        expected_pass=False),
]

all_claims = correct_claims + incorrect_claims
print(f"Total labeled claims: {len(all_claims)}")
print(f"  Correct (should pass): {len(correct_claims)}")
print(f"  Incorrect (should fail): {len(incorrect_claims)}")

### Step 3: Sweep $\theta_{\mathrm{sim}}$ (Plausibility Threshold)

The plausibility threshold controls how close two entities must be
on the manifold for a fact claim to pass. Larger values are more
lenient (accept more); smaller values are stricter.

In [6]:
calibrator = ThresholdCalibrator(store, device=str(DEVICE))

# Sweep theta_sim
theta_sim_values = [1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0]
sim_results = calibrator.sweep(
    param_name="theta_sim",
    values=theta_sim_values,
    test_claims=all_claims,
)

print("=== theta_sim Sweep ===")
print(f"{'theta_sim':>10s} {'accuracy':>9s} {'FA':>4s} {'FR':>4s}")
print("-" * 30)
for r in sim_results:
    print(f"{r.config.theta_sim:10.2f} {r.accuracy:9.4f} "
          f"{r.false_accepts:4d} {r.false_rejects:4d}")

=== theta_sim Sweep ===
 theta_sim  accuracy   FA   FR
------------------------------
      1.00    0.8333    2    0
      1.20    0.8333    2    0
      1.40    0.8333    2    0
      1.50    0.8333    2    0
      1.60    0.8333    2    0
      1.80    0.8333    2    0
      2.00    0.8333    2    0


### Step 4: Sweep $\theta_{\mathrm{margin}}$ (Numeric Tolerance)

The numeric tolerance controls how much deviation from the stored
value is acceptable for a numeric claim. Larger values accept
approximate matches; smaller values demand near-exact agreement.

In [7]:
# Sweep theta_margin (numeric tolerance)
theta_margin_values = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7]
margin_results = calibrator.sweep(
    param_name="theta_margin",
    values=theta_margin_values,
    test_claims=all_claims,
)

print("=== theta_margin Sweep ===")
print(f"{'theta_margin':>13s} {'accuracy':>9s} {'FA':>4s} {'FR':>4s}")
print("-" * 33)
for r in margin_results:
    print(f"{r.config.theta_margin:13.3f} {r.accuracy:9.4f} "
          f"{r.false_accepts:4d} {r.false_rejects:4d}")

=== theta_margin Sweep ===
 theta_margin  accuracy   FA   FR
---------------------------------
        0.010    1.0000    0    0
        0.050    0.9167    1    0
        0.100    0.8333    2    0
        0.200    0.8333    2    0
        0.300    0.8333    2    0
        0.500    0.8333    2    0
        0.700    0.8333    2    0


### Step 5: Visualize Accuracy vs Threshold

We plot the accuracy curve for each parameter sweep to identify
the operating region where the verifier balances false accepts
and false rejects.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- theta_sim ---
ax = axes[0]
sim_acc = [r.accuracy for r in sim_results]
sim_fa = [r.false_accepts for r in sim_results]
sim_fr = [r.false_rejects for r in sim_results]

ax.plot(theta_sim_values, sim_acc, "bo-", linewidth=2, label="Accuracy")
ax2 = ax.twinx()
ax2.bar([v - 0.03 for v in theta_sim_values], sim_fa,
        width=0.05, alpha=0.5, color="red", label="False Accepts")
ax2.bar([v + 0.03 for v in theta_sim_values], sim_fr,
        width=0.05, alpha=0.5, color="orange", label="False Rejects")
ax.set_xlabel(r"$\theta_{\mathrm{sim}}$")
ax.set_ylabel("Accuracy", color="blue")
ax2.set_ylabel("Error Count", color="red")
ax.set_title(r"Plausibility Threshold ($\theta_{\mathrm{sim}}$)")
ax.legend(loc="lower left")
ax2.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# --- theta_margin ---
ax = axes[1]
margin_acc = [r.accuracy for r in margin_results]
margin_fa = [r.false_accepts for r in margin_results]
margin_fr = [r.false_rejects for r in margin_results]

ax.plot(theta_margin_values, margin_acc, "go-", linewidth=2, label="Accuracy")
ax3 = ax.twinx()
ax3.bar([v - 0.02 for v in theta_margin_values], margin_fa,
        width=0.03, alpha=0.5, color="red", label="False Accepts")
ax3.bar([v + 0.02 for v in theta_margin_values], margin_fr,
        width=0.03, alpha=0.5, color="orange", label="False Rejects")
ax.set_xlabel(r"$\theta_{\mathrm{margin}}$")
ax.set_ylabel("Accuracy", color="green")
ax3.set_ylabel("Error Count", color="red")
ax.set_title(r"Numeric Tolerance ($\theta_{\mathrm{margin}}$)")
ax.legend(loc="lower left")
ax3.legend(loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch19_threshold_sweeps.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ch19_threshold_sweeps.png")

Saved: ch19_threshold_sweeps.png


### Step 6: Grid Search and Sensitivity Analysis

We run a factorial grid search over both $\theta_{\mathrm{sim}}$
and $\theta_{\mathrm{margin}}$ simultaneously, then compute
sensitivity scores to determine which parameter has the greater
impact on verification accuracy.

In [9]:
# Grid search: theta_sim x theta_margin
grid_results = calibrator.grid_search(
    param_grid={
        "theta_sim": [1.0, 1.2, 1.5, 1.8, 2.0],
        "theta_margin": [0.01, 0.1, 0.3, 0.5],
    },
    test_claims=all_claims,
)

print(f"Grid search: {len(grid_results)} configurations evaluated")
print(f"\n{calibrator.results_table(grid_results)}")

# Best configuration
best = calibrator.best_config(grid_results)
print("\n=== Best Configuration ===")
print(f"  theta_sim:    {best.config.theta_sim}")
print(f"  theta_margin: {best.config.theta_margin}")
print(f"  accuracy:     {best.accuracy}")
print(f"  false_accepts: {best.false_accepts}")
print(f"  false_rejects: {best.false_rejects}")

Grid search: 20 configurations evaluated

 theta_sim  theta_margin  theta_esc  accuracy   FA   FR  calls
--------------------------------------------------------------
      1.00         0.010       0.50    1.0000    0    0     12
      1.00         0.100       0.50    0.8333    2    0     12
      1.00         0.300       0.50    0.8333    2    0     12
      1.00         0.500       0.50    0.8333    2    0     12
      1.20         0.010       0.50    1.0000    0    0     12
      1.20         0.100       0.50    0.8333    2    0     12
      1.20         0.300       0.50    0.8333    2    0     12
      1.20         0.500       0.50    0.8333    2    0     12
      1.50         0.010       0.50    1.0000    0    0     12
      1.50         0.100       0.50    0.8333    2    0     12
      1.50         0.300       0.50    0.8333    2    0     12
      1.50         0.500       0.50    0.8333    2    0     12
      1.80         0.010       0.50    1.0000    0    0     12
      1.80   

In [10]:
# Sensitivity analysis
sensitivity = calibrator.sensitivity_analysis(grid_results)

print("=== Sensitivity Analysis ===")
print(f"{'Parameter':>15s} {'Impact Score':>13s}")
print("-" * 30)
for param, score in sorted(sensitivity.items(), key=lambda x: -x[1]):
    bar = "#" * int(score * 50)
    print(f"{param:>15s} {score:13.4f}  {bar}")

print(f"\nHighest impact: {max(sensitivity, key=sensitivity.get)}")
print(f"Lowest impact:  {min(sensitivity, key=sensitivity.get)}")

=== Sensitivity Analysis ===
      Parameter  Impact Score
------------------------------
   theta_margin        0.1667  ########
      theta_sim        0.0000  
      theta_esc        0.0000  
          d_max        0.0000  
         budget        0.0000  

Highest impact: theta_margin
Lowest impact:  theta_sim


In [11]:
# Visualize grid search as heatmap
sim_vals = sorted(set(r.config.theta_sim for r in grid_results))
mar_vals = sorted(set(r.config.theta_margin for r in grid_results))

acc_matrix = np.zeros((len(mar_vals), len(sim_vals)))
for r in grid_results:
    i = mar_vals.index(r.config.theta_margin)
    j = sim_vals.index(r.config.theta_sim)
    acc_matrix[i, j] = r.accuracy

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(acc_matrix, cmap="RdYlGn", aspect="auto",
               vmin=0.4, vmax=1.0)
ax.set_xticks(range(len(sim_vals)))
ax.set_xticklabels([f"{v:.1f}" for v in sim_vals])
ax.set_yticks(range(len(mar_vals)))
ax.set_yticklabels([f"{v:.2f}" for v in mar_vals])
ax.set_xlabel(r"$\theta_{\mathrm{sim}}$")
ax.set_ylabel(r"$\theta_{\mathrm{margin}}$")
ax.set_title("Grid Search: Verification Accuracy")

# Annotate cells
for i in range(len(mar_vals)):
    for j in range(len(sim_vals)):
        ax.text(j, i, f"{acc_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=9,
                color="white" if acc_matrix[i, j] < 0.6 else "black")

plt.colorbar(im, ax=ax, label="Accuracy")
plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch19_grid_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ch19_grid_heatmap.png")

Saved: ch19_grid_heatmap.png


### Classification Threshold Summary

The sweep and grid search reveal how each threshold parameter
affects verification accuracy on the Iris classification domain.
We record the best-performing configuration for comparison with
the regression domain in Lab 19.2.

In [12]:
# Record best Iris (classification) thresholds
iris_best = calibrator.best_config(grid_results)
print("=== Best Iris (Classification) Thresholds ===")
print(f"  theta_sim:    {iris_best.config.theta_sim}")
print(f"  theta_margin: {iris_best.config.theta_margin}")
print(f"  theta_esc:    {iris_best.config.theta_esc}")
print(f"  accuracy:     {iris_best.accuracy}")
print(f"  false_accepts: {iris_best.false_accepts}")
print(f"  false_rejects: {iris_best.false_rejects}")

=== Best Iris (Classification) Thresholds ===
  theta_sim:    1.0
  theta_margin: 0.01
  theta_esc:    0.5
  accuracy:     1.0
  false_accepts: 0
  false_rejects: 0


---

## Lab 19.2 --- Transfer to California Housing (Regression)

The governance architecture --- contracts, policies, claim types, GMS
primitives --- remains unchanged. Only three things change:

1. The **knowledge document** shifts from classification to regression.
2. The **dataset** shifts from Iris to California Housing.
3. The **threshold vector** is recalibrated for the new domain.

### Step 1: Ingest the Regression Knowledge Document

In [13]:
# --- Build the California Housing knowledge store ---
REG_MD_PATH = "governed_agent/knowledge/california_housing_domain.md"

reg_config = DocGMSConfig(
    store_path="governance_monograph/lab_stores/ch19_california",
    geometry=GeometryConfig(m=128),
    train=TrainConfig(epochs=200, device=str(DEVICE)),
)

reg_store = GMSExpertStore(reg_config, device=DEVICE)
if not reg_store.load():
    print("Building regression store from scratch (200 epochs)...")
    result = ingest_document(reg_store, REG_MD_PATH, llm=None,
                             config=reg_config, device=DEVICE)
    reg_store.save()

reg_stats = reg_store.stats()
print("\n=== California Housing Knowledge Store ===")
print(f"  Entities:  {reg_stats['entities']}")
print(f"  Relations: {reg_stats['relations']}")
print(f"  Triples:   {reg_stats['triples']}")
print(f"  ENM:       {reg_stats['enm_entries']}")

Building regression store from scratch (200 epochs)...

Ingesting: governed_agent/knowledge/california_housing_domain.md
  Converted to markdown: 8089 chars
  Parsed: 0 ENM, 84 triples, 4 phase encoders

  First document — building GMS from scratch...
  GMS entities:  69
  GMS relations: 8
  GMS triples:   84

Training GMS: 69 entities, 8 relations, 84 triples
  Config: d_v=128, m=64, epochs=200, batch=512, neg=64
  Device: cuda
  Epoch    1/200  Loss: 7.8187  (L_pos=1.5579, L_neg=0.0116)


  Epoch   20/200  Loss: 7.7984  (L_pos=1.5520, L_neg=0.0154)


  Epoch   40/200  Loss: 7.7573  (L_pos=1.5458, L_neg=0.0114)


  Epoch   60/200  Loss: 7.7334  (L_pos=1.5395, L_neg=0.0143)


  Epoch   80/200  Loss: 7.6931  (L_pos=1.5333, L_neg=0.0107)


...

  Epoch  120/200  Loss: 7.6397  (L_pos=1.5208, L_neg=0.0143)


  Epoch  140/200  Loss: 7.6043  (L_pos=1.5145, L_neg=0.0127)


  Epoch  160/200  Loss: 7.5751  (L_pos=1.5083, L_neg=0.0135)


  Epoch  180/200  Loss: 7.5522  (L_pos=1.5020, L_neg=0.0168)


  Epoch  200/200  Loss: 7.5119  (L_pos=1.4958, L_neg=0.0132)

  Populating ENM register...
  ENM populated: 0 entries
  Store saved to governance_monograph/lab_stores/ch19_california

  Ingestion complete: 69 entities, 84 triples, 0 ENM entries (2.1s)
  Store saved to governance_monograph/lab_stores/ch19_california

=== California Housing Knowledge Store ===
  Entities:  69
  Relations: 8
  Triples:   84
  ENM:       0


### Step 2: Load California Housing and Train Regression Models

In [ ]:
# --- Load California Housing via the governed tool interface ---
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.tools import execute_tool

state = SessionState()
display, raw = execute_tool(
    "load_dataset",
    {"name": "california_housing", "builtin": "california_housing"},
    state,
)
print(display)
print(f"\nTask type: {raw['task_type']}")
print(f"Features: {raw['feature_names']}")

In [15]:
# --- Train five regression models ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

ds = state.datasets["california_housing"]
X_tr = ds["X_train"].values
X_te = ds["X_test"].values
y_tr = ds["y_train"].values
y_te = ds["y_test"].values

reg_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=0.01, random_state=42),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=100, random_state=42),
}

reg_metrics = {}
print("=== Regression Model Training ===")
print(f"{'Model':>30s} {'R2':>8s} {'MSE':>10s} {'MAE':>8s}")
print("-" * 60)

for name, model in reg_models.items():
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2 = round(r2_score(y_te, y_pred), 4)
    mse = round(mean_squared_error(y_te, y_pred), 4)
    mae = round(mean_absolute_error(y_te, y_pred), 4)
    reg_metrics[name] = {"r2": r2, "mse": mse, "mae": mae}
    print(f"{name:>30s} {r2:8.4f} {mse:10.4f} {mae:8.4f}")

=== Regression Model Training ===
                         Model       R2        MSE      MAE
------------------------------------------------------------
              LinearRegression   0.5758     0.5559   0.5332
                         Ridge   0.5759     0.5558   0.5332
                         Lasso   0.5845     0.5444   0.5363


         RandomForestRegressor   0.8051     0.2554   0.3275


     GradientBoostingRegressor   0.7756     0.2940   0.3716


In [16]:
# --- Record regression metrics in the regression GMS store ---
reg_learner = RuntimeLearner(reg_store, reg_config.learn)

reg_triples = []
for model_name, mvals in reg_metrics.items():
    for metric_name, value in mvals.items():
        reg_triples.append(ExtractedTriple(
            head=model_name, relation=f"has_{metric_name}",
            tail=f"{value}", numeric_value=value))
    reg_triples.append(ExtractedTriple(
        head=model_name, relation="trained_on",
        tail="california_housing"))

reg_result = reg_learner.learn_triples(reg_triples, device=DEVICE)
reg_store.save()
print(f"RuntimeLearner accepted: {reg_result.accepted}")
print(f"  Triples added: {reg_result.n_triples_added}")
print(f"  Training queued: {reg_result.training_queued}")

  Store saved to governance_monograph/lab_stores/ch19_california
RuntimeLearner accepted: True
  Triples added: 20
  Training queued: True


### Step 3: Verify Regression Claims

We build a set of labeled claims for the regression domain.
The claim types are identical to classification (numeric,
threshold, fact, comparison) --- only the metric names and
expected values change.

In [ ]:
from knowlytix.harness.governance.verifier import ClaimVerifier

# Build verifier on regression store
reg_verifier = ClaimVerifier(reg_store, device=str(DEVICE))

# Dynamically construct claims from actual metric values
gb_r2 = reg_metrics["GradientBoostingRegressor"]["r2"]
rf_r2 = reg_metrics["RandomForestRegressor"]["r2"]
lr_r2 = reg_metrics["LinearRegression"]["r2"]
gb_mse = reg_metrics["GradientBoostingRegressor"]["mse"]
rf_mse = reg_metrics["RandomForestRegressor"]["mse"]

# --- Correct regression claims ---
reg_correct = [
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="GradientBoostingRegressor trained on california_housing",
                    metadata={"head": "GradientBoostingRegressor",
                              "relation": "trained_on",
                              "tail": "california_housing"}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text=f"GradientBoosting R2 is {gb_r2}",
                    metadata={"category": "has_r2",
                              "entity_id": f"GradientBoostingRegressor/{gb_r2}",
                              "claimed_value": gb_r2}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text=f"RandomForest R2 is {rf_r2}",
                    metadata={"category": "has_r2",
                              "entity_id": f"RandomForestRegressor/{rf_r2}",
                              "claimed_value": rf_r2}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.THRESHOLD,
                    text="GradientBoosting R2 meets 0.70 minimum",
                    metadata={"category": "has_r2",
                              "entity_id": f"GradientBoostingRegressor/{gb_r2}",
                              "operator": "ge", "limit": 0.70}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="RandomForestRegressor trained on california_housing",
                    metadata={"head": "RandomForestRegressor",
                              "relation": "trained_on",
                              "tail": "california_housing"}),
        expected_pass=True),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text=f"LinearRegression R2 is {lr_r2}",
                    metadata={"category": "has_r2",
                              "entity_id": f"LinearRegression/{lr_r2}",
                              "claimed_value": lr_r2}),
        expected_pass=True),
]

# --- Incorrect regression claims ---
reg_incorrect = [
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="GradientBoosting R2 is 0.99 (WRONG)",
                    metadata={"category": "has_r2",
                              "entity_id": f"GradientBoostingRegressor/{gb_r2}",
                              "claimed_value": 0.99}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="RandomForest MSE is 0.01 (WRONG)",
                    metadata={"category": "has_mse",
                              "entity_id": f"RandomForestRegressor/{rf_mse}",
                              "claimed_value": 0.01}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="Lasso is a tree model (WRONG)",
                    metadata={"head": "Lasso",
                              "relation": "is_type",
                              "tail": "tree_model"}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.THRESHOLD,
                    text="LinearRegression R2 exceeds 0.99 (WRONG)",
                    metadata={"category": "has_r2",
                              "entity_id": f"LinearRegression/{lr_r2}",
                              "operator": "ge", "limit": 0.99}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.FACT,
                    text="SVM trained on california_housing (WRONG)",
                    metadata={"head": "SVM",
                              "relation": "trained_on",
                              "tail": "california_housing"}),
        expected_pass=False),
    LabeledClaim(
        claim=Claim(type=ClaimType.NUMERIC,
                    text="Ridge R2 is 0.05 (WRONG)",
                    metadata={"category": "has_r2",
                              "entity_id": f"Ridge/{reg_metrics['Ridge']['r2']}",
                              "claimed_value": 0.05}),
        expected_pass=False),
]

reg_all_claims = reg_correct + reg_incorrect
print(f"Regression labeled claims: {len(reg_all_claims)}")
print(f"  Correct (should pass): {len(reg_correct)}")
print(f"  Incorrect (should fail): {len(reg_incorrect)}")

In [18]:
# --- Verify regression claims with default thresholds ---
reg_verdict = reg_verifier.verify_all([lc.claim for lc in reg_all_claims])

print(reg_verifier.format_overall(reg_verdict))
print()

for i, v in enumerate(reg_verdict.claim_verdicts):
    expected = reg_all_claims[i].expected_pass
    match = "OK" if v.passed == expected else "MISMATCH"
    print(f"  [{match:8s}] {reg_verifier.format_verdict(v)}")
    print()

=== Overall Verdict: FAIL ===
  Passed: 6/12 (50.0%)
  Severity distribution: {'info': 6, 'error': 5, 'critical': 1}

  [OK      ]   [+] fact         | PASS | score=0.900 | severity=info     | GradientBoostingRegressor trained on california_housing
      evidence: (GradientBoostingRegressor, trained_on, california_housing) verified via query_triples (exact graph match)

  [OK      ]   [+] numeric      | PASS | score=1.000 | severity=info     | GradientBoosting R2 is 0.7756
      evidence: lookup_enm(has_r2, GradientBoostingRegressor/0.7756) = 0.775600, claimed = 0.775600, rel_diff = 0.000000 <= 0.01

  [OK      ]   [+] numeric      | PASS | score=1.000 | severity=info     | RandomForest R2 is 0.8051
      evidence: lookup_enm(has_r2, RandomForestRegressor/0.8051) = 0.805100, claimed = 0.805100, rel_diff = 0.000000 <= 0.01

  [OK      ]   [+] threshold    | PASS | score=0.878 | severity=info     | GradientBoosting R2 meets 0.70 minimum
      evidence: lookup_enm(has_r2, GradientBoosting

### Step 4: Recalibrate Thresholds for Regression

We run the same grid search on the regression claims to find
the best threshold configuration for the new domain.

In [19]:
# --- Grid search on regression domain ---
reg_calibrator = ThresholdCalibrator(reg_store, device=str(DEVICE))

reg_grid_results = reg_calibrator.grid_search(
    param_grid={
        "theta_sim": [1.0, 1.2, 1.5, 1.8, 2.0],
        "theta_margin": [0.01, 0.1, 0.3, 0.5],
    },
    test_claims=reg_all_claims,
)

print(f"Regression grid search: {len(reg_grid_results)} configurations")
print(f"\n{reg_calibrator.results_table(reg_grid_results)}")

reg_best = reg_calibrator.best_config(reg_grid_results)
print("\n=== Best California Housing (Regression) Thresholds ===")
print(f"  theta_sim:    {reg_best.config.theta_sim}")
print(f"  theta_margin: {reg_best.config.theta_margin}")
print(f"  accuracy:     {reg_best.accuracy}")
print(f"  false_accepts: {reg_best.false_accepts}")
print(f"  false_rejects: {reg_best.false_rejects}")

Regression grid search: 20 configurations

 theta_sim  theta_margin  theta_esc  accuracy   FA   FR  calls
--------------------------------------------------------------
      1.00         0.010       0.50    1.0000    0    0     12
      1.00         0.100       0.50    1.0000    0    0     12
      1.00         0.300       0.50    0.8333    2    0     12
      1.00         0.500       0.50    0.7500    3    0     12
      1.20         0.010       0.50    1.0000    0    0     12
      1.20         0.100       0.50    1.0000    0    0     12
      1.20         0.300       0.50    0.8333    2    0     12
      1.20         0.500       0.50    0.7500    3    0     12
      1.50         0.010       0.50    1.0000    0    0     12
      1.50         0.100       0.50    1.0000    0    0     12
      1.50         0.300       0.50    0.8333    2    0     12
      1.50         0.500       0.50    0.7500    3    0     12
      1.80         0.010       0.50    1.0000    0    0     12
      1.80  

In [20]:
# --- Sensitivity analysis for regression ---
reg_sensitivity = reg_calibrator.sensitivity_analysis(reg_grid_results)

print("=== Regression Sensitivity Analysis ===")
print(f"{'Parameter':>15s} {'Impact Score':>13s}")
print("-" * 30)
for param, score in sorted(reg_sensitivity.items(), key=lambda x: -x[1]):
    bar = "#" * int(score * 50)
    print(f"{param:>15s} {score:13.4f}  {bar}")

=== Regression Sensitivity Analysis ===
      Parameter  Impact Score
------------------------------
   theta_margin        0.2500  ############
      theta_sim        0.0000  
      theta_esc        0.0000  
          d_max        0.0000  
         budget        0.0000  


### Step 5: Compare Classification vs Regression Thresholds

We now compare the optimal thresholds and sensitivity scores
across the two domains. The key finding: the governance architecture
is identical --- only the threshold values differ.

In [21]:
# --- Side-by-side comparison ---
print("=" * 60)
print("  CROSS-DOMAIN THRESHOLD COMPARISON")
print("=" * 60)
print(f"\n{'Parameter':>15s} {'Iris (clf)':>12s} {'CalHousing (reg)':>18s}")
print("-" * 48)
print(f"{'theta_sim':>15s} {iris_best.config.theta_sim:12.2f} "
      f"{reg_best.config.theta_sim:18.2f}")
print(f"{'theta_margin':>15s} {iris_best.config.theta_margin:12.3f} "
      f"{reg_best.config.theta_margin:18.3f}")
print(f"{'theta_esc':>15s} {iris_best.config.theta_esc:12.2f} "
      f"{reg_best.config.theta_esc:18.2f}")
print(f"\n{'accuracy':>15s} {iris_best.accuracy:12.4f} "
      f"{reg_best.accuracy:18.4f}")
print(f"{'false_accepts':>15s} {iris_best.false_accepts:12d} "
      f"{reg_best.false_accepts:18d}")
print(f"{'false_rejects':>15s} {iris_best.false_rejects:12d} "
      f"{reg_best.false_rejects:18d}")

# Sensitivity comparison
cls_sensitivity = calibrator.sensitivity_analysis(grid_results)
print(f"\n{'':>15s} {'--- Sensitivity ---':>32s}")
for p in ["theta_sim", "theta_margin", "theta_esc"]:
    c_val = cls_sensitivity.get(p, 0.0)
    r_val = reg_sensitivity.get(p, 0.0)
    print(f"{p:>15s} {c_val:12.4f} {r_val:18.4f}")

  CROSS-DOMAIN THRESHOLD COMPARISON

      Parameter   Iris (clf)   CalHousing (reg)
------------------------------------------------
      theta_sim         1.00               1.00
   theta_margin        0.010              0.010
      theta_esc         0.50               0.50

       accuracy       1.0000             1.0000
  false_accepts            0                  0
  false_rejects            0                  0

                             --- Sensitivity ---
      theta_sim       0.0000             0.0000
   theta_margin       0.1667             0.2500
      theta_esc       0.0000             0.0000


In [22]:
# --- Bar chart: classification vs regression thresholds ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Threshold values comparison
ax = axes[0]
params = [r"$\theta_{sim}$", r"$\theta_{margin}$", r"$\theta_{esc}$"]
cls_vals = [iris_best.config.theta_sim, iris_best.config.theta_margin,
            iris_best.config.theta_esc]
reg_vals = [reg_best.config.theta_sim, reg_best.config.theta_margin,
            reg_best.config.theta_esc]

x = np.arange(len(params))
w = 0.35
ax.bar(x - w/2, cls_vals, w, label="Iris (classification)", color="steelblue")
ax.bar(x + w/2, reg_vals, w, label="CalHousing (regression)", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(params)
ax.set_ylabel("Threshold Value")
ax.set_title("Optimal Threshold Values by Domain")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Sensitivity comparison
ax = axes[1]
cls_sens = [cls_sensitivity.get(p, 0) for p in ["theta_sim", "theta_margin", "theta_esc"]]
reg_sens = [reg_sensitivity.get(p, 0) for p in ["theta_sim", "theta_margin", "theta_esc"]]

ax.bar(x - w/2, cls_sens, w, label="Iris (classification)", color="steelblue")
ax.bar(x + w/2, reg_sens, w, label="CalHousing (regression)", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(params)
ax.set_ylabel("Sensitivity (accuracy range)")
ax.set_title("Parameter Sensitivity by Domain")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch19_domain_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ch19_domain_comparison.png")

Saved: ch19_domain_comparison.png


### Architecture Portability Summary

The following components remain **unchanged** across domains:

| Component | Classification | Regression |
|-----------|---------------|------------|
| ClaimTypes | 5 types | 5 types (same) |
| GMS primitives | score_triple, lookup_enm, check_holonomy | same |
| Verification logic | ClaimVerifier | same code |
| Severity classification | 4 levels | same |
| Calibration method | grid search | same |

Only the **threshold vector** and **knowledge document** change.

In [23]:
# --- Final summary table ---
print("=" * 65)
print("  ARCHITECTURE PORTABILITY EVIDENCE")
print("=" * 65)
print(f"\n  {'Item':30s} {'Classification':>15s} {'Regression':>15s}")
print(f"  {'-'*30} {'-'*15} {'-'*15}")
print(f"  {'Dataset':30s} {'Iris':>15s} {'CalHousing':>15s}")
print(f"  {'Task type':30s} {'classification':>15s} {'regression':>15s}")
print(f"  {'Models trained':30s} {3:>15d} {5:>15d}")
print(f"  {'Knowledge store entities':30s} {stats['entities']:>15d} "
      f"{reg_stats['entities']:>15d}")
print(f"  {'Knowledge store triples':30s} {stats['triples']:>15d} "
      f"{reg_stats['triples']:>15d}")
print(f"  {'Claim types used':30s} {'5 (same)':>15s} {'5 (same)':>15s}")
print(f"  {'GMS primitives used':30s} {'3 (same)':>15s} {'3 (same)':>15s}")
print(f"  {'Verifier code changed':30s} {'No':>15s} {'No':>15s}")
print(f"  {'Best accuracy':30s} {iris_best.accuracy:>15.4f} "
      f"{reg_best.accuracy:>15.4f}")
print(f"  {'Best theta_sim':30s} {iris_best.config.theta_sim:>15.2f} "
      f"{reg_best.config.theta_sim:>15.2f}")
print(f"  {'Best theta_margin':30s} {iris_best.config.theta_margin:>15.3f} "
      f"{reg_best.config.theta_margin:>15.3f}")
print("\n  Conclusion: architecture portable, only thresholds recalibrated.")

  ARCHITECTURE PORTABILITY EVIDENCE

  Item                            Classification      Regression
  ------------------------------ --------------- ---------------
  Dataset                                   Iris      CalHousing
  Task type                       classification      regression
  Models trained                               3               5
  Knowledge store entities                   244              69
  Knowledge store triples                    370              84
  Claim types used                      5 (same)        5 (same)
  GMS primitives used                   3 (same)        3 (same)
  Verifier code changed                       No              No
  Best accuracy                           1.0000          1.0000
  Best theta_sim                            1.00            1.00
  Best theta_margin                        0.010           0.010

  Conclusion: architecture portable, only thresholds recalibrated.


---

## Exercises

### Exercise 19.1 --- Minimum $\theta_{\mathrm{margin}}$ for Zero False Accepts

Find the minimum value of $\theta_{\mathrm{margin}}$ that achieves
zero false acceptances on a held-out set of 50 adversarial claims.
Use binary search to narrow the range efficiently.

**Hint:** Start with a fine sweep of `theta_margin` over
`np.linspace(0.001, 0.5, 50)` and find the smallest value where
`false_accepts == 0`.

In [24]:
# Exercise 19.1 -- your code here
# fine_values = np.linspace(0.001, 0.5, 50).tolist()
# fine_results = calibrator.sweep("theta_margin", fine_values, all_claims)
# for r in fine_results:
#     if r.false_accepts == 0:
#         print(f"Min theta_margin for zero FA: {r.config.theta_margin:.4f}")
#         print(f"  Accuracy: {r.accuracy}, FR: {r.false_rejects}")
#         break

### Exercise 19.2 --- Auto-Calibrator

Build an `auto_calibrate(calibrator, claims, target_recall=0.95)`
function that runs the agent on a validation dataset, measures
gate accuracy and adjusts thresholds to maximize precision at
the specified recall level.

**Hint:** For each grid point, compute precision = correct / (correct + FA)
and recall = correct_pass / total_should_pass. Filter to configurations
meeting the recall target, then pick the one with highest precision.

In [25]:
# Exercise 19.2 -- your code here
# def auto_calibrate(calibrator, claims, target_recall=0.95):
#     grid = calibrator.grid_search(
#         {"theta_sim": np.linspace(0.8, 2.5, 10).tolist(),
#          "theta_margin": np.linspace(0.001, 0.5, 10).tolist()},
#         claims)
#     n_should_pass = sum(1 for c in claims if c.expected_pass)
#     best_precision, best_cfg = 0.0, None
#     for r in grid:
#         tp = n_should_pass - r.false_rejects
#         recall = tp / n_should_pass if n_should_pass > 0 else 0
#         precision = tp / (tp + r.false_accepts) if (tp + r.false_accepts) > 0 else 0
#         if recall >= target_recall and precision > best_precision:
#             best_precision = precision
#             best_cfg = r
#     return best_cfg
#
# result = auto_calibrate(calibrator, all_claims, target_recall=0.95)
# print(f"Best: {result.config}, precision={...}")

### Exercise 19.3 --- Store Quality vs Governance Accuracy

What is the minimum GMS store quality (measured by mean
`score_triple` on known facts) needed for governance to work
reliably? Build stores with varying epoch counts (10, 50, 100,
200, 500) and plot governance accuracy vs store quality.

**Hint:** Train multiple stores with different epoch counts.
For each, compute the mean geodesic distance on known-true triples
and run the calibrator. Plot `mean_geodesic` vs `best_accuracy`.

In [26]:
# Exercise 19.3 -- your code here
# epoch_counts = [10, 50, 100, 200, 500]
# qualities, accuracies = [], []
# for epochs in epoch_counts:
#     cfg = DocGMSConfig(
#         store_path=f"governance_monograph/lab_stores/ch19_quality_{epochs}",
#         geometry=GeometryConfig(m=128),
#         train=TrainConfig(epochs=epochs, device=str(DEVICE)))
#     s = GMSExpertStore(cfg, device=DEVICE)
#     ingest_document(s, MD_PATH, llm=None, config=cfg, device=DEVICE)
#     # ... record metrics, run calibrator ...
#     qualities.append(mean_geodesic)
#     accuracies.append(best.accuracy)
# plt.plot(qualities, accuracies, 'bo-')
# plt.xlabel('Mean Geodesic (store quality)')
# plt.ylabel('Best Governance Accuracy')
# plt.show()

---
## Lab 19.3 — Audit-Defensible Calibration Hardening

The labs so far showed how to calibrate threshold vectors via DOE methodology and how the calibration transfers across domains. An auditor's next questions:

1. Does the calibration result **reproduce across runs**? → deterministic `CalibrationSnapshot` fingerprint.
2. Does an **independent implementation** reach the same optimum? → alt-validator (sorted-pair lookup vs primary's `max(key=...)`).
3. How do thresholds **drift across domains**? → `compare_calibrations_across_domains`.
4. Does the bundle **authenticate its producer**? → HMAC-signed `CalibrationBundle` with three refusal paths.

Library: `governed_agent/calibration_hardening/` (7 modules).

### Lab 19.3 §1 — CalibrationSnapshot for Iris and California

In [ ]:
from knowlytix.harness.governance.calibration_hardening import (
    build_calibration_snapshot,
)

iris_snap = build_calibration_snapshot(
    suite_id="lab-19-3-iris",
    domain="iris",
    grid_keys=sorted(grid_results[0].config.__dict__.keys()),
    sweep_results=grid_results,
    optimum=iris_best,
)
print(iris_snap.summary())
print(f"\nIris fingerprint: {iris_snap.fingerprint()}")

cal_snap = build_calibration_snapshot(
    suite_id="lab-19-3-california",
    domain="california_housing",
    grid_keys=sorted(reg_grid_results[0].config.__dict__.keys()),
    sweep_results=reg_grid_results,
    optimum=reg_best,
)
print(f"\n{cal_snap.summary()}")
print(f"California fingerprint: {cal_snap.fingerprint()}")


### Lab 19.3 §2 — Cross-domain agreement

In [ ]:
from knowlytix.harness.governance.calibration_hardening import (
    compare_calibrations_across_domains,
)

cross = compare_calibrations_across_domains(iris_snap, cal_snap)
print(cross.summary())


### Lab 19.3 §3 — Build signed CalibrationBundle

In [ ]:
from knowlytix.harness.governance.calibration_hardening import (
    CalibrationOperatingPoint, build_calibration_bundle,
    bind_calibration_from_bundle, assert_calibration_alt_agreement,
)

# Alt-validator first
sweep_lookup = {
    "lab-19-3-iris": grid_results,
    "lab-19-3-california": reg_grid_results,
}
alt = assert_calibration_alt_agreement(
    [iris_snap, cal_snap], sweep_lookup=sweep_lookup,
)
print(alt.summary())
print()

# Build the Iris bundle (one bundle per domain in deployment)
op = CalibrationOperatingPoint(
    min_accuracy=max(iris_snap.optimum.accuracy - 0.02, 0.5),
    require_alt_agreement=True,
)
bundle = build_calibration_bundle(
    snapshot=iris_snap,
    alt_agreement_rate=alt.agreement_rate,
    operating_point=op,
)
print(f"Iris bundle id: {bundle.bundle_id}")
print(f"signing_method: {bundle.signing_method}")
print(f"signature: {bundle.signature[:24]}...")

# Bind back - 3 refusal paths
bind_calibration_from_bundle(bundle, snapshot=iris_snap)
print("\nbound: snapshot fingerprint match + optimum match + SLO satisfied")


---
## Lab 19.4 — Production Hardening

Five additional gaps closed:

1. FSM invariants over 19 calibration states (M1–M5)
2. Independent alt-validator: sorted-pair re-derivation of optimum
3. k-of-n multi-signer workflow (reuses Ch 12 `policy_multisign` API)
4. Microbenchmarks: snapshot.fingerprint p99 ~30μs, alt_best_config p99 ~7μs
5. Model card at `gms_agent_builder_ext/MODEL_CARD_calibration_bundle.md`

Runbook: `gms_agent_builder_ext/lab19_calibration_runbook.py`. Ship gate: `scripts/gate_calibration_bundle.py`.

### Lab 19.4 §1 — FSM invariants

In [ ]:
from knowlytix.harness.governance.calibration_hardening import verify_calibration_invariants

inv = verify_calibration_invariants()
print(inv.summary())
assert inv.all_pass, "FSM invariant violation -- ship blocker"


### Lab 19.4 §2 — Multi-signer + microbenchmarks

In [ ]:
from knowlytix.harness.governance import attach_signatures, sign_for, verify_quorum

keys = {
    "mrm_lead":           "deployment-mrm-lead-key-32-bytes!!",
    "compliance_officer": "deployment-compliance-key-32-bytes!",
    "platform_owner":     "deployment-platform-key-32-bytes!!",
}
sigs = [sign_for(bundle, sid, k) for sid, k in keys.items()]
attach_signatures(bundle, sigs)
quorum = verify_quorum(bundle, keys, k=2)
print(quorum.summary())
print()

from knowlytix.harness.governance.calibration_hardening import (
    bench_alt_best_config, bench_calibration_fingerprint,
)
print(bench_calibration_fingerprint(iris_snap, n_calls=2000, warmup=100).summary())
print(bench_alt_best_config(grid_results, n_calls=2000, warmup=100).summary())
